# LA Wildfire NDVI Comparison

Starting on January 7 2025, a series of large wildfires hit the LA County area. The years prior had been particularly wet, so grasses and chaparral could accumulate in the mountains and foothills around the city. The dry season at the end of 2024 dried out much of the vegetation and left significant fodder for wildfires (science.nasa.gov). When strong, hot winds hit the area in January, all that was needed was a spark and the fires eventually covered 37,000 acres. Total property and capital losses are estimated to range between $76 and $131 billion, with additional losses in wages for local businesses, higher insurance premiums for Californians, and an impact on the already stressed housing market (anderson.ucla.edu).

The following code was my attempt to look at the impact of the January 2025 wildfires on NDVI in LA county. I first downloaded the county boundaries and NDVI data from 2019 to 2025 in the area. I then made a plot comparing the difference between average NDVI from 2019 to the end of 2024 and from the start of 2025 to present day. 

In [1]:
# Import libraries
import json
import os
import pathlib

import earthpy
import geopandas as gpd

import hvplot.pandas
import hvplot.xarray
import rioxarray as rxr
import xarray as xr

from datetime import datetime
import matplotlib


In [2]:
# Define project
project = earthpy.Project("LA Fire Vegetation", dirname='la_fire_vegetation_data')

# Load in the boundary data
boundary_gdf = gpd.read_file(project.project_dir / 'boundary')

In [3]:
# We now want to get the boundary data in the correct 
# projection and plot the boundary on a map to see how it looks.
boundary_gdf = boundary_gdf.to_crs(epsg=4326)

boundary_gdf.hvplot(
    geo=True,
    tiles='EsriImagery',
    frame_width=500,
    legend=False,
    fill_color=None,
    edge_color='white',
)


:Overlay
   .WMTS.I     :WMTS   [Longitude,Latitude]
   .Polygons.I :Polygons   [Longitude,Latitude]

In [4]:
# Get a sorted list of NDVI tif file paths
ndvi_paths = sorted(list(project.project_dir.rglob('*EVI*.tif')))

# Display the first and last three files paths to check the pattern
ndvi_paths[:3], ndvi_paths[-3:]

# Clean the NDVI data and make the files sortable by date
ndvi_das = []

doy_start = -25
doy_end = -18

for ndvi_path in ndvi_paths:
    file_str = str(ndvi_path)
    doy_string = file_str[-25:-18]  # assuming this slice is correct
    date = datetime.strptime(doy_string, "%Y%j")

    # Open lazily with chunks so it doesn't load everything at once
    da = rxr.open_rasterio(
        ndvi_path,
        masked=True,
        chunks={"y": 1024, "x": 1024}  # tweak chunk size if needed
    )

    # Drop band dimension if it's just 1 band
    da = da.squeeze(drop=True)

    # Use float32 instead of float64 to cut memory in half
    da = (da.astype("float32") / 10000.0)

    da.name = "NDVI"

    # Add a time dimension in one go
    da = da.expand_dims(time=[date])

    ndvi_das.append(da)

# Only now concatenate along time
ndvi_stack = xr.concat(ndvi_das, dim="time")

# Reproject boundary to match MODIS data
modis_crs = ndvi_stack.rio.crs
boundary_modis = boundary_gdf.to_crs(modis_crs)


In [5]:
# Combine NDVI images from all dates
ndvi_time_series = xr.combine_by_coords(ndvi_das)

In [6]:
# Compute the difference in NDVI before and after land ownership change

# Define time ranges before the change in land ownership(2001-2011) 
# and after (2012-2022)
pre_start = 2019177
pre_end = 2025001
post_start = 2025017
post_end = 2025177

# Convert these time ranges into datetime format
pre_start_dt = datetime.strptime(str(pre_start), "%Y%j")
pre_end_dt   = datetime.strptime(str(pre_end), "%Y%j")
post_start_dt = datetime.strptime(str(post_start), "%Y%j")
post_end_dt   = datetime.strptime(str(post_end), "%Y%j")
    

# Compute the mean NDVI before the change
ndvi_pre = ndvi_time_series["NDVI"].sel({'time': slice(pre_start_dt, pre_end_dt)})
ndvi_pre_mean = ndvi_pre.mean(dim='time', skipna=True)

# Compute the mean NDVI after the change
ndvi_post = ndvi_time_series["NDVI"].sel({'time': slice(post_start_dt, post_end_dt)})
ndvi_post_mean = ndvi_post.mean(dim='time', skipna=True)
print(ndvi_post_mean)

# Calculate the change in NDVI by subtracting the values after the change
# from the values before the change 
ndvi_diff = ndvi_post_mean - ndvi_pre_mean

# Plot the difference
(
    ndvi_diff.hvplot(x='x', y='y', cmap='PRGn', geo=True)
    *
    boundary_gdf.hvplot(geo=True, fill_color=None, line_color='black')
)

<xarray.DataArray 'NDVI' (y: 1343, x: 3241)> Size: 17MB
dask.array<mean_agg-aggregate, shape=(1343, 3241), dtype=float32, chunksize=(1024, 1024), chunktype=numpy.ndarray>
Coordinates:
  * x            (x) float64 26kB -1.128e+07 -1.128e+07 ... -1.053e+07
  * y            (y) float64 11kB 3.907e+06 3.907e+06 ... 3.596e+06 3.596e+06
    spatial_ref  int64 8B 0


:Overlay
   .Image.I    :Image   [x,y]   (NDVI)
   .Polygons.I :Polygons   [Longitude,Latitude]

## Conclusions
When comparing my map to a map from nasa.gov showing summer 2025 NDVI anomaly (percent difference from 1991-2020 average), my map looks quite similar! I was quite surprised, considering that I hadn't used a 10 year average as my "pre-event" NDVI. There is a good chunk of decreased NDVI in the Angeles National Forest, which was a major site of the fire. I don't know much about other wildfires that could have been in the area since the beginning of 2025, but I would not be surprised if those were the other spots on the map where we see decreased NDVI values.

(The map as it is has an error in the coordinate reference system, where the boundary reference system is different from that of the NDVI. I could not figure out how to fix this efficiently. )

## References
https://www.anderson.ucla.edu/about/centers/ucla-anderson-forecast/economic-impact-los-angeles-wildfires

https://science.nasa.gov/earth/earth-observatory/fuel-for-california-fires-153896/